# Data Transformations: Bronze → Silver
### AdventureWorks — Performance-Optimized & Best-Practice Edition

This notebook ingests raw CSV files from the **Bronze** layer, applies cleansing /
enrichment transformations, and writes curated **Delta** tables to the **Silver**
layer.

**Enhancements over the original version:**
- Parameterized, secret-based configuration (no hard-coded paths/creds)
- Explicit schemas on read (no `inferSchema`) — faster, safer, deterministic
- Spark/Delta performance tuning (AQE, shuffle partitions, auto-compaction)
- A single reusable, error-handled load/write pipeline instead of copy-pasted code
- Delta Lake instead of Parquet (ACID, schema evolution, `MERGE`, time travel)
- Partitioning + `OPTIMIZE`/`Z-ORDER` on high-volume tables
- Data-quality checks (row counts, null %, duplicate keys) before every write
- Caching for DataFrames that are reused, with explicit `unpersist`
- Logging instead of ad-hoc `display()` calls sprinkled through the pipeline


## 1. Imports & Spark/Delta Performance Configuration

Setting these once at the top of the job (rather than relying on cluster
defaults) makes the notebook's performance characteristics explicit and
reproducible across clusters/environments.

| Setting | Why |
|---|---|
| `spark.sql.adaptive.enabled` | Adaptive Query Execution — re-optimizes joins/shuffles at runtime using real statistics |
| `spark.sql.adaptive.coalescePartitions.enabled` | Merges small shuffle partitions automatically → avoids the small-files problem |
| `spark.sql.shuffle.partitions` | `auto`/tuned value instead of the 200-partition default, which is wasteful for small/medium data |
| `spark.databricks.delta.optimizeWrite.enabled` | Auto-compacts files during write → avoids small-file problem without a manual `OPTIMIZE` |
| `spark.databricks.delta.autoCompact.enabled` | Background compaction for tables written incrementally |
| `spark.sql.parquet.compression.codec` | `snappy` — good balance of speed vs. size for analytical workloads |


In [ ]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import *
from pyspark.sql.types import *
import logging

# ---- Logging (replaces scattered display()/print() debugging) -------------
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("bronze_to_silver")

# ---- Performance-oriented Spark configuration ------------------------------
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "auto")  # let AQE decide instead of a fixed 200
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")

logger.info("Spark session configured for adaptive execution and Delta auto-optimization.")

## 2. Configuration

Hard-coding the storage account name in every path (as in the original
notebook) makes the notebook environment-specific and leaks infrastructure
details into business logic. Using **widgets** (parameterization) + a
**secret scope** for credentials lets the same notebook run unchanged across
dev/test/prod and be safely scheduled as a job.


In [ ]:
dbutils.widgets.text("storage_account", "shoaibadlsdev", "ADLS Storage Account")
dbutils.widgets.text("bronze_container", "bronze", "Bronze Container")
dbutils.widgets.text("silver_container", "silver", "Silver Container")

STORAGE_ACCOUNT   = dbutils.widgets.get("storage_account")
BRONZE_CONTAINER  = dbutils.widgets.get("bronze_container")
SILVER_CONTAINER  = dbutils.widgets.get("silver_container")

BRONZE_PATH = f"abfss://{BRONZE_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
SILVER_PATH = f"abfss://{SILVER_CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Auth: prefer a secret-scope-backed service principal / SAS over notebook-level
# credentials baked into paths. Uncomment and adapt to your workspace's scope:
# spark.conf.set(
#     f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net",
#     dbutils.secrets.get(scope="adls-scope", key="sp-client-secret")
# )

logger.info(f"Bronze path: {BRONZE_PATH}")
logger.info(f"Silver path: {SILVER_PATH}")

## 3. Data Access
Quick sanity check that the bronze container is reachable and lists the expected source folders.

In [ ]:
display(dbutils.fs.ls(BRONZE_PATH))

## 4. Explicit Schemas (instead of `inferSchema`)

The original notebook used `option("inferSchema", "true")` for every table.
`inferSchema` forces Spark to **read the entire file twice** (once to infer
types, once to actually load) — on large files this roughly doubles I/O and
job time. It's also non-deterministic across schema drift (a stray blank
value can silently flip a column from `IntegerType` to `StringType`).

Defining `StructType` schemas up front:
- Reads the file **once**
- Fails fast (and loudly) on unexpected structure instead of silently
  mis-typing a column
- Documents the contract of each bronze source in code


In [ ]:
schema_calendar = StructType([
    StructField("Date", DateType(), True),
])

schema_customers = StructType([
    StructField("CustomerKey", IntegerType(), True),
    StructField("Prefix", StringType(), True),
    StructField("FirstName", StringType(), True),
    StructField("LastName", StringType(), True),
    StructField("BirthDate", DateType(), True),
    StructField("MaritalStatus", StringType(), True),
    StructField("Gender", StringType(), True),
    StructField("EmailAddress", StringType(), True),
    StructField("AnnualIncome", DoubleType(), True),
    StructField("TotalChildren", IntegerType(), True),
    StructField("EducationLevel", StringType(), True),
    StructField("Occupation", StringType(), True),
    StructField("HomeOwner", StringType(), True),
])

schema_product_categories = StructType([
    StructField("ProductCategoryKey", IntegerType(), True),
    StructField("CategoryName", StringType(), True),
])

schema_product_subcategories = StructType([
    StructField("ProductSubcategoryKey", IntegerType(), True),
    StructField("SubcategoryName", StringType(), True),
    StructField("ProductCategoryKey", IntegerType(), True),
])

schema_products = StructType([
    StructField("ProductKey", IntegerType(), True),
    StructField("ProductSubcategoryKey", IntegerType(), True),
    StructField("ProductSKU", StringType(), True),
    StructField("ProductName", StringType(), True),
    StructField("ModelName", StringType(), True),
    StructField("ProductDescription", StringType(), True),
    StructField("ProductColor", StringType(), True),
    StructField("ProductSize", StringType(), True),
    StructField("ProductStyle", StringType(), True),
    StructField("ProductCost", DoubleType(), True),
    StructField("ProductPrice", DoubleType(), True),
])

schema_returns = StructType([
    StructField("ReturnDate", DateType(), True),
    StructField("TerritoryKey", IntegerType(), True),
    StructField("ProductKey", IntegerType(), True),
    StructField("ReturnQuantity", IntegerType(), True),
])

schema_territories = StructType([
    StructField("SalesTerritoryKey", IntegerType(), True),
    StructField("Region", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Continent", StringType(), True),
])

schema_sales = StructType([
    StructField("OrderDate", DateType(), True),
    StructField("StockDate", DateType(), True),
    StructField("OrderNumber", StringType(), True),
    StructField("ProductKey", IntegerType(), True),
    StructField("CustomerKey", IntegerType(), True),
    StructField("TerritoryKey", IntegerType(), True),
    StructField("OrderLineItem", IntegerType(), True),
    StructField("OrderQuantity", IntegerType(), True),
])

logger.info("Explicit schemas defined for all 8 bronze sources.")

## 5. Reusable Load / Validate / Write Functions

The original notebook repeated the same `spark.read.format("csv")...` and
`.write.format("parquet")...` boilerplate **eight times**. Duplicated code is
a maintenance and correctness risk (a fix to one table's read options has to
be manually copy-pasted to the other seven). Wrapping the pattern in
functions:
- Keeps the pipeline DRY and easy to extend to new sources
- Centralizes error handling / logging
- Makes data-quality checks a mandatory step of every load rather than an
  optional afterthought


In [ ]:
def load_bronze_csv(file_name: str, schema: StructType) -> DataFrame:
    """Read a bronze CSV with an explicit schema (single-pass, no inferSchema).

    Using `mode="FAILFAST"` surfaces malformed rows immediately instead of
    silently nulling them out or corrupting downstream aggregates.
    """
    path = f"{BRONZE_PATH}/{file_name}"
    try:
        df = (
            spark.read.format("csv")
            .option("header", "true")
            .schema(schema)
            .option("mode", "FAILFAST")
            .load(path)
        )
        logger.info(f"Loaded '{file_name}' ({df.count():,} rows).")
        return df
    except Exception as e:
        logger.error(f"Failed to load bronze source '{file_name}': {e}")
        raise


def profile_dataframe(df: DataFrame, name: str) -> None:
    """Lightweight data-quality gate: row count, duplicate count, null % per column.

    Cheap enough to run on every table, but catches broken upstream extracts
    (e.g. a bronze file that suddenly loads as 0 rows, or a key column that
    is unexpectedly 100% null) before they silently propagate to Silver/Gold.
    """
    total = df.count()
    dupes = total - df.dropDuplicates().count()
    null_report = df.select([
        (count(when(col(c).isNull(), c)) / total).alias(c) for c in df.columns
    ]).collect()[0].asDict()
    high_null_cols = {c: round(pct * 100, 1) for c, pct in null_report.items() if pct > 0.2}

    logger.info(f"[{name}] rows={total:,} | duplicate_rows={dupes:,}")
    if high_null_cols:
        logger.warning(f"[{name}] columns >20% null: {high_null_cols}")
    if total == 0:
        raise ValueError(f"[{name}] loaded 0 rows — aborting pipeline.")


def write_silver_delta(
    df: DataFrame,
    table_name: str,
    partition_by: list | None = None,
    optimize_zorder: list | None = None,
) -> None:
    """Write a curated DataFrame to Silver as Delta, with optional partitioning
    and post-write OPTIMIZE/Z-ORDER for read performance.

    Delta (vs. the original notebook's Parquet) buys us:
    - ACID overwrite semantics (no partially-written table if the job fails mid-write)
    - Schema enforcement/evolution (`mergeSchema`) instead of silent column drift
    - Time travel for auditing/rollback
    - `OPTIMIZE` / `Z-ORDER` for compaction and data-skipping on large tables
    """
    path = f"{SILVER_PATH}/{table_name}"
    writer = (
        df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    writer.save(path)
    logger.info(f"Wrote '{table_name}' to Silver as Delta at {path}"
                + (f" (partitioned by {partition_by})" if partition_by else ""))

    if optimize_zorder:
        cols = ", ".join(optimize_zorder)
        backtick = chr(96)
        spark.sql(f"OPTIMIZE delta.{backtick}{path}{backtick} ZORDER BY ({cols})")
        logger.info(f"Optimized + Z-ORDERed '{table_name}' by ({cols}).")

## 6. Data Loading
All eight bronze sources loaded through the single, schema-enforced, logged loader.

In [ ]:
df_cal     = load_bronze_csv("AdventureWorks_Calendar", schema_calendar)
df_cus     = load_bronze_csv("AdventureWorks_Customers", schema_customers)
df_procat  = load_bronze_csv("AdventureWorks_Product_Categories", schema_product_categories)
df_subcat  = load_bronze_csv("AdventureWorks_Product_Subcategories", schema_product_subcategories)
df_pro     = load_bronze_csv("AdventureWorks_Products", schema_products)
df_ret     = load_bronze_csv("AdventureWorks_Returns", schema_returns)
df_ter     = load_bronze_csv("AdventureWorks_Territories", schema_territories)

# Sales is a multi-file wildcard load (AdventureWorks_Sales*). Wildcards are fine
# here because all files share the schema we defined — that assumption is now
# enforced (FAILFAST) instead of assumed via inferSchema.
df_sales   = load_bronze_csv("AdventureWorks_Sales*", schema_sales)

## 7. Data-Quality Gate
Run before any Silver write — fails fast on empty loads, surfaces high-null columns and duplicate rows.

In [ ]:
for _df, _name in [
    (df_cal, "Calendar"), (df_cus, "Customers"), (df_procat, "Product_Categories"),
    (df_subcat, "Product_Subcategories"), (df_pro, "Products"), (df_ret, "Returns"),
    (df_ter, "Territories"), (df_sales, "Sales"),
]:
    profile_dataframe(_df, _name)

## 8. Transformations
Each table's transformation, immediately followed by its Silver write. Tables that are wide/high-cardinality get `OPTIMIZE ... ZORDER` for downstream query performance.

### AdventureWorks_Calendar

In [ ]:
df_cal = df_cal.withColumn("Month", month(col("Date"))) \
               .withColumn("Year", year(col("Date")))

display(df_cal.limit(5))

In [ ]:
# Small dimension table — partitioning would create excessive small files, so we
# write it as a single unpartitioned Delta table.
write_silver_delta(df_cal, "AdventureWorks_Calendar")

### AdventureWorks_Customers

In [ ]:
df_cus = df_cus.withColumn("Fullname", concat_ws(" ", col("Prefix"), col("FirstName"), col("LastName")))
display(df_cus.limit(5))

In [ ]:
write_silver_delta(df_cus, "AdventureWorks_Customers", optimize_zorder=["CustomerKey"])

### Product_Subcategories

In [ ]:
write_silver_delta(df_subcat, "Product_Subcategories")

### AdventureWorks_Products

In [ ]:
df_pro = df_pro.withColumn("ProductSKU", split(col("ProductSKU"), "-")[0]) \
               .withColumn("ProductName", split(col("ProductName"), " ")[0])

display(df_pro.limit(5))

In [ ]:
write_silver_delta(df_pro, "AdventureWorks_Products", optimize_zorder=["ProductKey"])

### AdventureWorks_Returns

In [ ]:
write_silver_delta(df_ret, "AdventureWorks_Returns", partition_by=["TerritoryKey"])

### AdventureWorks_Territories

In [ ]:
write_silver_delta(df_ter, "AdventureWorks_Territories")

### Product_Categories

In [ ]:
write_silver_delta(df_procat, "AdventureWorks_Product_Categories")

### Sales
This is the largest, most frequently-queried table, so it gets the most performance attention: caching (it is both written and analyzed below), partitioning by the natural query dimension, and `OPTIMIZE ... ZORDER` on the join keys.

In [ ]:
df_sales = (
    df_sales
    .withColumn("StockDate", to_timestamp(col("StockDate")))
    .withColumn("OrderNumber", regexp_replace(col("OrderNumber"), "S", "T"))
    .withColumn("Multiply", col("OrderLineItem") * col("OrderQuantity"))
    .withColumn("OrderYear", year(col("OrderDate")))
)

# Cached because df_sales is reused for both the Silver write and the
# analysis section below — without caching, Spark would re-read and
# re-transform the source data for each downstream action.
df_sales.cache()
display(df_sales.limit(5))

In [ ]:
write_silver_delta(
    df_sales,
    "AdventureWorks_Sales",
    partition_by=["OrderYear"],           # aligns with typical "sales by year" query patterns
    optimize_zorder=["ProductKey", "CustomerKey"],  # common join keys → data-skipping on joins
)

## 9. Sales Analysis
Reuses the cached `df_sales` (no re-read from disk).

In [ ]:
display(df_sales.groupBy("OrderDate").agg(count("OrderNumber").alias("Total_Orders")).orderBy("OrderDate"))

In [ ]:
display(df_procat)

In [ ]:
display(df_ter)

## 10. Cleanup
Release cached memory once the pipeline is done with it — leaving DataFrames cached for the lifetime of the cluster is a common cause of executor memory pressure on shared/job clusters.

In [ ]:
df_sales.unpersist()
logger.info("Pipeline complete. Bronze -> Silver load, transform, and write finished successfully.")